# Generating Graphs

In [11]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
import re

# Load Data

In [12]:
filepath_finetuned = "../data/results_eval_mteb_finetuned.csv"
filepath_baseline = "../data/results_eval_mteb.csv"

results_finetuned = pd.read_csv(filepath_finetuned)
results_baseline = pd.read_csv(filepath_baseline)

# Pre-processing

In [13]:
task_type_map = {
    "Assin2STS": "STS",
    "SICK-BR-STS": "STS",
    "STSBenchmarkMultilingualSTS": "STS",
    'MassiveIntentClassification': "Classification",
    'MultiHateClassification': "Classification",
    'BrazilianToxicTweetsClassification': "Classification",
    'HateSpeechPortugueseClassification': "Classification",
    'TweetSentimentClassification': "Classification",
    'MultiLongDocReranking': "Reranking",
    'WikipediaRerankingMultilingual': "Reranking",
    'XGlueWPRReranking': "Reranking",
    'WebFAQRetrieval': "Retrieval",
    'MultiLongDocRetrieval': "Retrieval",
    'WikipediaRetrievalMultilingual': "Retrieval",
}

max_emb_dim_map = {
    'paraphrase-multilingual-MiniLM-L12-v2': 384,
    'Legal-BERTimbau-sts-large-ma-v3': 1024,
    'Qwen3-Embedding-0.6B': 1024,
    'bert-base-portuguese-cased': 1024,

    'e5-large-matryoshka-sts-pt': 1024,
    'BERTimbau-matryoshka-sts-pt': 1024,
    'NeoBERTugues-matryoshka-sts-pt': 768,
    'mmBERT-base-matryoshka-sts-pt': 768,
}

In [ ]:
results_all = pd.concat([results_finetuned, results_baseline], ignore_index=True)
results_all['task_type'] = results_all['task_name'].map(task_type_map)
results_all['model_raw_name'] = results_all['model_name'].apply(lambda x: x.split("/")[-1])
results_all['max_emb_size'] = results_all['model_raw_name'].map(max_emb_dim_map)
results_all['embedding_dim_full'] = results_all['embedding_dim'].combine_first(results_all['max_emb_size']).astype(int)
results_all.head()

,model_name,embedding_dim,task_name,main_score,task_type,model_raw_name,max_emb_size,embedding_dim_full
0,iara-project/e5-large-matryoshka-sts-pt,NaN,Assin2STS,0.841035,STS,e5-large-matryoshka-sts-pt,1024,1024
1,iara-project/e5-large-matryoshka-sts-pt,NaN,SICK-BR-STS,0.917959,STS,e5-large-matryoshka-sts-pt,1024,1024
2,iara-project/e5-large-matryoshka-sts-pt,NaN,STSBenchmarkMultilingualSTS,0.879975,STS,e5-large-matryoshka-sts-pt,1024,1024
3,iara-project/e5-large-matryoshka-sts-pt,NaN,MassiveIntentClassification,0.662842,Classification,e5-large-matryoshka-sts-pt,1024,1024
4,iara-project/e5-large-matryoshka-sts-pt,NaN,MultiHateClassification,0.666700,Classification,e5-large-matryoshka-sts-pt,1024,1024


# Export Graphs

In [17]:
output_dir = "../graphs"
os.makedirs(output_dir, exist_ok=True)

def sanitize_filename(name):
    """Remove caracteres problemáticos para nome de arquivo"""
    name = str(name)
    name = re.sub(r"[^\w\s-]", "", name)  # remove caracteres especiais
    name = re.sub(r"\s+", "_", name)      # espaços -> _
    return name.lower()

sns.set_theme(style="whitegrid", context="talk")

# =========================
# 1) Preparação dos dados
# =========================

df_plot = results_all.copy()

df_plot["num_dims"] = df_plot["embedding_dim"].fillna(df_plot["embedding_dim_full"])
df_plot["num_dims"] = pd.to_numeric(df_plot["num_dims"], errors="coerce")
df_plot["main_score"] = pd.to_numeric(df_plot["main_score"], errors="coerce")

df_plot = df_plot.dropna(subset=["num_dims", "main_score", "model_raw_name"])


# =========================================================
# 2) Gráficos por task_name
# =========================================================

task_names = sorted(df_plot["task_name"].dropna().unique())

for task in task_names:
    df_task = (
        df_plot[df_plot["task_name"] == task]
        .groupby(["task_name", "model_raw_name", "num_dims"], as_index=False)["main_score"]
        .mean()
    )

    x_order = sorted(df_task["num_dims"].unique(), reverse=True)

    plt.figure(figsize=(12, 6))
    sns.lineplot(
        data=df_task,
        x="num_dims",
        y="main_score",
        hue="model_raw_name",
        marker="o",
        sort=False
    )

    plt.title(f"Main Score por dimensão - Task: {task}")
    plt.xlabel("Número de dimensões")
    plt.ylabel("Main Score")
    plt.xticks(x_order)
    plt.gca().invert_xaxis()
    plt.legend(
        title="Model Name",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        fontsize=9,          # ↓ tamanho dos labels
        title_fontsize=10,   # ↓ título da legenda
        markerscale=0.8      # ↓ tamanho dos marcadores na legenda
    )
    plt.tight_layout()

    filename = f"task_{sanitize_filename(task)}.png"
    plt.savefig(os.path.join(output_dir, filename))
    plt.close()


# =========================================================
# 3) Gráficos por task_type (média)
# =========================================================

df_task_type = (
    df_plot
    .groupby(["task_type", "model_raw_name", "num_dims"], as_index=False)["main_score"]
    .mean()
)

task_types = sorted(df_task_type["task_type"].dropna().unique())

for ttype in task_types:
    df_tt = df_task_type[df_task_type["task_type"] == ttype].copy()
    x_order = sorted(df_tt["num_dims"].unique(), reverse=True)

    plt.figure(figsize=(12, 6))
    sns.lineplot(
        data=df_tt,
        x="num_dims",
        y="main_score",
        hue="model_raw_name",
        marker="o",
        sort=False
    )

    plt.title(f"Média de Main Score - Task Type: {ttype}")
    plt.xlabel("Número de dimensões")
    plt.ylabel("Main Score médio")
    plt.xticks(x_order)
    plt.gca().invert_xaxis()
    plt.legend(
        title="Model Name",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        fontsize=9,          # ↓ tamanho dos labels
        title_fontsize=10,   # ↓ título da legenda
        markerscale=0.8      # ↓ tamanho dos marcadores na legenda
    )
    plt.tight_layout()

    filename = f"task_type_{sanitize_filename(ttype)}.png"
    plt.savefig(os.path.join(output_dir, filename))
    plt.close()


# =========================================================
# 4) Gráfico global (média por model_raw_name)
# =========================================================

df_global = (
    df_plot
    .groupby(["model_raw_name", "num_dims"], as_index=False)["main_score"]
    .mean()
)

x_order = sorted(df_global["num_dims"].unique(), reverse=True)

plt.figure(figsize=(12, 6))
sns.lineplot(
    data=df_global,
    x="num_dims",
    y="main_score",
    hue="model_raw_name",
    marker="o",
    sort=False
)

plt.title("Média global de Main Score por dimensão")
plt.xlabel("Número de dimensões")
plt.ylabel("Main Score médio global")
plt.xticks(x_order)
plt.gca().invert_xaxis()
plt.legend(
        title="Model Name",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        fontsize=9,          # ↓ tamanho dos labels
        title_fontsize=10,   # ↓ título da legenda
        markerscale=0.8      # ↓ tamanho dos marcadores na legenda
    )
plt.tight_layout()

filename = "global_media_main_score_por_dimensao.png"
plt.savefig(os.path.join(output_dir, filename))
plt.close()